# 완성된 작품을 사이트에 등록

영어 원문·영상 프롬프트 JSONL, 선택한 한국어 원문, 완성된 MP4를 함께 보냅니다. 모든 업로드 뒤 사이트 등록 성공까지 확인합니다. 기존 프롬프트의 대사 유무를 변경하거나 모델을 실행하지 않습니다.

같은 폴더의 README.md를 먼저 읽으세요. 파일은 **Jupyter 커널 서버**에 있어야 하며 인코딩을 끝내고 파일을 닫은 뒤 실행합니다.


이 전달본은 세 작품 재생성용으로 설정되어 있습니다. Jupyter 작업 폴더를 이 `integrations/content-publisher/`로 맞추고, 설정 셀의 `WORK_SLUG`와 `VIDEOS_DIR`를 선택하세요. 한국어 파일과 작품 정보는 자동으로 대응합니다. 기존 영상 교체가 켜져 있으므로 **새 프롬프트로 다시 만든 MP4만** 지정하세요. 생성 시 장면 길이는 `../../data/short_stories/{slug}.audio.json`의 `scenes`를 따릅니다.


In [ ]:
%pip install -r requirements.txt


In [ ]:
import os
from getpass import getpass
from clipit_publish import dedicated_session, load_private_config, prepare_publication, publish_completed

_private_config_loaded = load_private_config(".env.content-publisher.json")


## 1. 작품과 파일 지정

생성자에게 받은 다섯 필드 JSONL을 사용합니다. MP4 폴더의 이름은 `1_1.mp4`, `1_2.mp4`처럼 장·장면 번호에 맞춥니다. 한국어 JSONL이 없으면 `None`을 유지합니다.

같은 폴더에 비공개 `.env.content-publisher.json`이 있으면 위 셀이 허용된 여섯 설정만 환경에 불러옵니다. 내용은 출력하지 않으며 일반 ZIP에 넣지 않습니다. POSIX 서버의 파일 권한은 0600이어야 합니다. 파일이 없으면 아래에서 숨김 입력으로 키를 받습니다.

새 영상으로 기존 장면을 교체할 때만 `REPLACE_VIDEOS`를 켭니다. 기존 번역을 수정할 때만 `REPLACE_TRANSLATIONS`를 켭니다. 영어 원문은 두 옵션으로도 덮어쓰지 않습니다.


In [ ]:
from pathlib import Path
import json

# Choose a key from ../../data/short_stories/books.json.
WORK_SLUG = "the-gift-of-the-magi"
# The completed MP4 files must exist on this Jupyter kernel server.
VIDEOS_DIR = "/path/to/completed/the-gift-of-the-magi"

repo_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                  if (p / "data/short_stories/books.json").is_file()), None)
if repo_root is None:
    raise RuntimeError("Open this notebook from the cloned Capstone_LLM repository.")
data_root = repo_root / "data/short_stories"
works = json.loads((data_root / "books.json").read_text(encoding="utf-8"))
work = works[WORK_SLUG]
SCRIPTS_PATH = str(data_root / f"{WORK_SLUG}.jsonl")
KOREAN_JSONL = str(data_root / f"{WORK_SLUG}.source-ko.jsonl")
SLUG, TITLE, TITLE_KO, AUTHOR = WORK_SLUG, work["title"], work["title_ko"], work["author"]
GENRE = "Short fiction"
CLIPIT_API_BASE = os.environ.get("CLIPIT_API_BASE", "https://cbnu-clip-it.vercel.app")
# These three works already have the older videos on the site.
REPLACE_VIDEOS = True
REPLACE_TRANSLATIONS = False

options = dict(slug=SLUG, title=TITLE, author=AUTHOR, title_ko=TITLE_KO, genre=GENRE,
               korean_path=KOREAN_JSONL, replace_videos=REPLACE_VIDEOS,
               replace_translations=REPLACE_TRANSLATIONS)


## 2. 로컬 파일 검증

이 셀은 키나 클라우드 연결 없이 파일 내용·장면 대응·완성 MP4를 검사합니다.


In [ ]:
prepared = prepare_publication(SCRIPTS_PATH, VIDEOS_DIR, **options)
print({"검증한 장면": len(prepared["videos"]), "한국어 원문이 있는 장면": prepared["translated_scenes"], "클라우드 호출": 0})


## 3. 담당자 전용 키 입력

원문·스크립트용 AWS 키와 영상용 AWS 키를 각각 넣습니다. 게시 토큰은 별도로 받은 `CONTENT_PUBLISH_TOKEN`입니다. 관리자 키나 DB 암호는 필요하지 않습니다. 숨김 입력값은 셀 코드나 출력에 저장하지 않습니다. 커널을 종료하면 다시 입력해야 합니다.


In [ ]:
script_session = dedicated_session(
    os.environ.get("CLIPIT_SCRIPT_ACCESS_KEY_ID") or getpass("원문·스크립트 업로드 Access Key ID: "),
    os.environ.get("CLIPIT_SCRIPT_SECRET_ACCESS_KEY") or getpass("원문·스크립트 업로드 Secret Access Key: "),
    role="script",
)
video_session = dedicated_session(
    os.environ.get("CLIPIT_VIDEO_ACCESS_KEY_ID") or getpass("영상 업로드 Access Key ID: "),
    os.environ.get("CLIPIT_VIDEO_SECRET_ACCESS_KEY") or getpass("영상 업로드 Secret Access Key: "),
    role="video",
)
publish_token = os.environ.get("CONTENT_PUBLISH_TOKEN") or getpass("CONTENT_PUBLISH_TOKEN: ")
print("두 업로드 전용 계정을 확인했습니다.")


## 4. 업로드하고 사이트에 등록

이 셀을 실행하면 실제로 저장하고 게시합니다. 완료될 때까지 파일을 수정하지 마세요. 기존 영어 원문과 다르거나 교체 옵션 없이 다른 영상이 있으면 사이트 변경을 거절합니다. 업로드 후 네트워크가 끊겨도 같은 파일로 다시 실행할 수 있습니다.


In [ ]:
receipt = publish_completed(
    SCRIPTS_PATH, VIDEOS_DIR, script_session=script_session, video_session=video_session,
    publish_token=publish_token, api_base=CLIPIT_API_BASE, **options,
)
print(receipt)


## 성공 확인

영수증의 `ok: True`, `registered_in_web: True`, `registered_scenes`를 확인합니다. 성공하면 작품 원문과 영상이 사이트에 연결됩니다. 게시한 작품을 열어 장면 대응과 재생도 확인하세요. 키나 토큰을 영수증과 함께 공유하지 않습니다.

사이트 연결 오류가 나면 README.md의 오류 코드별 다음 행동을 확인합니다. 완성 영상을 다시 생성할 필요 없이 동일 파일로 재시도할 수 있습니다.
